# OAuth-Protected MCP Server Targets with PKCE Authentication

## Overview

This tutorial demonstrates how to connect AgentCore Gateway to GitHub's MCP server using:
- **Inbound auth**: PKCE (browser login, no client secret needed)
- **Outbound auth**: Authorization Code Grant (3LO) for GitHub access

The user authenticates via Cognito in the browser, gets an access token, and uses it to call the gateway. The gateway handles the 3LO flow with GitHub transparently.

### Architecture

```
User → Browser Login (PKCE) → Cognito Access Token
     → Gateway (Bearer token, validated by allowedClients)
     → Gateway checks token vault for GitHub 3LO token
     → If missing: returns elicitation URL → user consents on GitHub
     → If cached: injects GitHub token → proxies to GitHub MCP server
```

### Why PKCE?

Without PKCE, the client would need the Cognito `client_secret` to exchange the authorization code for a token — meaning every user or MCP client would need access to the secret. PKCE eliminates this by using a one-time code verifier instead, making it safe for public clients (desktop apps, CLI tools, IDE extensions) where secrets cannot be stored securely.

Note: The GitHub `client_id` and `client_secret` in the configuration below are only used at **setup time** to register the credential provider with AgentCore Identity. They are never exposed to the client — the gateway handles the outbound OAuth exchange with GitHub entirely server-side.

### Prerequisites

- AWS credentials configured
- GitHub OAuth App created (Client ID + Client Secret)
- Python 3.10+

In [ ]:
# Install dependencies
!pip install -q boto3 requests

In [ ]:
import boto3
import json
import time
import os
import sys
import requests
import subprocess
import hashlib
import base64
import secrets
import urllib.parse
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
REGION = os.environ.get('AWS_REGION', 'us-west-2')

print(f"Region: {REGION}")
print(f"Timestamp: {timestamp}")
print("✓ Libraries imported")

## Configuration

Fill in your GitHub OAuth App credentials and desired settings.

In [ ]:
# === CONFIGURATION — Fill in your values ===

# GitHub OAuth App credentials (from https://github.com/settings/developers)
# These are needed ONLY at setup time to register the GitHub credential provider
# with AgentCore Identity. They are NOT used by the client at runtime — the gateway
# and AgentCore Identity handle the OAuth exchange with GitHub on your behalf.
GITHUB_CLIENT_ID = ""       # From GitHub OAuth App
GITHUB_CLIENT_SECRET = ""   # From GitHub OAuth App

# Gateway settings
GATEWAY_NAME = f"oauth-3lo-gateway-{timestamp}"
MCP_VERSION = "2025-11-25"
CALLBACK_URL = "http://localhost:3000/callback"
PKCE_CALLBACK_URL = "http://localhost:3001/callback"

# Cognito settings (public client — no secret needed for PKCE)
COGNITO_POOL_NAME = f"oauth-3lo-pool-{timestamp}"
COGNITO_DOMAIN_PREFIX = f"oauth-3lo-{timestamp}"
TEST_USER_EMAIL = "testuser@example.com"
TEST_USER_PASSWORD = "TestPass1!"

assert GITHUB_CLIENT_ID, "Set GITHUB_CLIENT_ID"
assert GITHUB_CLIENT_SECRET, "Set GITHUB_CLIENT_SECRET"

print("✓ Configuration set")


In [ ]:
# Initialize AWS clients
cognito = boto3.client('cognito-idp', region_name=REGION)
agentcore_cp = boto3.client('bedrock-agentcore-control', region_name=REGION)
agentcore_dp = boto3.client('bedrock-agentcore', region_name=REGION)

# State tracking
state = {}
print("✓ AWS clients initialized")

## Step 1: Create Cognito User Pool

We create a Cognito User Pool with two app clients:
- **Public client** (no secret, PKCE) — for browser-based login
- **Test user** — for testing the flow

In [ ]:
# Create User Pool
pool = cognito.create_user_pool(
    PoolName=COGNITO_POOL_NAME,
    AutoVerifiedAttributes=['email'],
    UsernameAttributes=['email'],
    Policies={'PasswordPolicy': {'MinimumLength': 8, 'RequireUppercase': True, 'RequireLowercase': True, 'RequireNumbers': True, 'RequireSymbols': True}},
    Schema=[{'Name': 'email', 'AttributeDataType': 'String', 'Required': True, 'Mutable': True}]
)
state['pool_id'] = pool['UserPool']['Id']
print(f"✓ User Pool: {state['pool_id']}")

# Create domain
cognito.create_user_pool_domain(UserPoolId=state['pool_id'], Domain=COGNITO_DOMAIN_PREFIX)
state['cognito_domain'] = COGNITO_DOMAIN_PREFIX
print(f"✓ Domain: {COGNITO_DOMAIN_PREFIX}.auth.{REGION}.amazoncognito.com")

# Create public client (PKCE, no secret)
public_client = cognito.create_user_pool_client(
    UserPoolId=state['pool_id'],
    ClientName=f"public-pkce-client-{timestamp}",
    GenerateSecret=False,
    ExplicitAuthFlows=['ALLOW_REFRESH_TOKEN_AUTH', 'ALLOW_USER_SRP_AUTH'],
    SupportedIdentityProviders=['COGNITO'],
    CallbackURLs=[CALLBACK_URL, PKCE_CALLBACK_URL],
    AllowedOAuthFlows=['code'],
    AllowedOAuthScopes=['openid', 'email', 'profile'],
    AllowedOAuthFlowsUserPoolClient=True
)
state['public_client_id'] = public_client['UserPoolClient']['ClientId']
print(f"✓ Public client: {state['public_client_id']} (PKCE, no secret)")

# Create test user
try:
    cognito.admin_create_user(
        UserPoolId=state['pool_id'], Username=TEST_USER_EMAIL,
        UserAttributes=[{'Name': 'email', 'Value': TEST_USER_EMAIL}, {'Name': 'email_verified', 'Value': 'true'}],
        TemporaryPassword=TEST_USER_PASSWORD, MessageAction='SUPPRESS'
    )
    cognito.admin_set_user_password(
        UserPoolId=state['pool_id'], Username=TEST_USER_EMAIL,
        Password=TEST_USER_PASSWORD, Permanent=True
    )
except Exception as e:
    print(f"User may already exist: {e}")
print(f"✓ Test user: {TEST_USER_EMAIL} / {TEST_USER_PASSWORD}")

# OIDC discovery URL
state['discovery_url'] = f"https://cognito-idp.{REGION}.amazonaws.com/{state['pool_id']}/.well-known/openid-configuration"
print(f"✓ Discovery: {state['discovery_url']}")

## Step 2: Create AgentCore Gateway

The gateway uses `allowedClients` to validate the `client_id` claim in Cognito access tokens. This works with PKCE (public client) because the access token contains `client_id` (not `aud`).

In [ ]:
# Create IAM role for Gateway with scoped permissions
iam = boto3.client('iam', region_name=REGION)
role_name = f'AgentCoreGatewayRole-{timestamp}'

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    }]
}

# Scoped policy for gateway operations
gateway_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Sid': 'AgentCoreGatewayAccess',
            'Effect': 'Allow',
            'Action': [
                'bedrock-agentcore:InvokeMcpTool',
                'bedrock-agentcore:GetResourceOauth2Token',
                'bedrock-agentcore:CompleteResourceTokenAuth',
                'bedrock-agentcore:GetWorkloadAccessToken',
                'bedrock-agentcore:GetWorkloadAccessTokenForJWT'
            ],
            'Resource': '*'
        },
        {
            'Sid': 'SecretsManagerAccess',
            'Effect': 'Allow',
            'Action': [
                'secretsmanager:GetSecretValue'
            ],
            'Resource': 'arn:aws:secretsmanager:*:*:secret:bedrock-agentcore-identity*'
        }
    ]
}

try:
    role = iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='IAM role for AgentCore Gateway with scoped permissions'
    )
    GATEWAY_ROLE_ARN = role['Role']['Arn']
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName='AgentCoreGatewayPolicy',
        PolicyDocument=json.dumps(gateway_policy)
    )
    print(f'✓ Created IAM role: {GATEWAY_ROLE_ARN}')
except iam.exceptions.EntityAlreadyExistsException:
    GATEWAY_ROLE_ARN = iam.get_role(RoleName=role_name)['Role']['Arn']
    print(f'✓ IAM role exists: {GATEWAY_ROLE_ARN}')

state['role_name'] = role_name
# Wait for IAM propagation
time.sleep(10)

In [ ]:
# Create gateway with allowedClients
gw = agentcore_cp.create_gateway(
    name=GATEWAY_NAME,
    description='OAuth-protected MCP server targets with PKCE inbound auth',
    roleArn=GATEWAY_ROLE_ARN,
    protocolType='MCP',
    protocolConfiguration={'mcp': {'supportedVersions': [MCP_VERSION]}},
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': state['discovery_url'],
            'allowedClients': [state['public_client_id']]
        }
    }
)
state['gateway_id'] = gw['gatewayId']
print(f"✓ Gateway created: {state['gateway_id']}")

# Wait for READY
for _ in range(24):
    status = agentcore_cp.get_gateway(gatewayIdentifier=state['gateway_id'])['status']
    if status == 'READY': break
    time.sleep(5)

gw_info = agentcore_cp.get_gateway(gatewayIdentifier=state['gateway_id'])
state['gateway_url'] = gw_info['gatewayUrl']
print(f"✓ Gateway READY: {state['gateway_url']}")

## Step 3: Create GitHub Credential Provider + Target

Register GitHub as an OAuth2 credential provider in AgentCore Identity, then create a 3LO MCP server target pointing to GitHub's MCP endpoint.

In [ ]:
# Create GitHub OAuth2 credential provider
provider = agentcore_cp.create_oauth2_credential_provider(
    name=f"github-3lo-{timestamp}",
    credentialProviderVendor='GithubOauth2',
    oauth2ProviderConfigInput={
        'githubOauth2ProviderConfig': {
            'clientId': GITHUB_CLIENT_ID,
            'clientSecret': GITHUB_CLIENT_SECRET
        }
    }
)
state['github_provider_name'] = f"github-3lo-{timestamp}"
state['github_provider_arn'] = provider['credentialProviderArn']
state['github_callback_url'] = provider['callbackUrl']
print(f"✓ GitHub provider: {state['github_provider_name']}")
print(f"  Callback URL: {state['github_callback_url']}")
print(f"")
print(f"⚠ UPDATE YOUR GITHUB OAUTH APP:")
print(f"  Set Authorization callback URL to: {state['github_callback_url']}")
print(f"  https://github.com/settings/developers")

In [ ]:
# Start callback server for admin auth (target creation may need it)
callback_proc = subprocess.Popen(
    [sys.executable, 'oauth2_callback_server.py', REGION, '3000', '/tmp/gateway-jwt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(1)

# Create GitHub MCP server target with 3LO
target = agentcore_cp.create_gateway_target(
    gatewayIdentifier=state['gateway_id'],
    name=f"github-mcp-3lo-{timestamp}",
    description='GitHub MCP server with 3LO outbound auth',
    targetConfiguration={'mcp': {'mcpServer': {'endpoint': 'https://api.githubcopilot.com/mcp'}}},
    credentialProviderConfigurations=[{
        'credentialProviderType': 'OAUTH',
        'credentialProvider': {
            'oauthCredentialProvider': {
                'providerArn': state['github_provider_arn'],
                'grantType': 'AUTHORIZATION_CODE',
                'defaultReturnUrl': CALLBACK_URL,
                'scopes': ['repo', 'user', 'workflow']
            }
        }
    }]
)
state['github_target_id'] = target['targetId']
state['github_target_name'] = f"github-mcp-3lo-{timestamp}"
print(f"✓ Target created: {state['github_target_id']} (status: {target['status']})")

# If CREATE_PENDING_AUTH, complete via console
if target['status'] == 'CREATE_PENDING_AUTH':
    print("")
    print("=" * 60)
    print("⚠️  ACTION REQUIRED: Admin Authorization Needed")
    print("=" * 60)
    print(f"")
    print(f"The target is in CREATE_PENDING_AUTH status.")
    print(f"You MUST authorize it in the AgentCore console before continuing.")
    print(f"")
    print(f"1. Go to the AgentCore console")
    print(f"2. Navigate to Gateway: {state['gateway_id']}")
    print(f"3. Click on the target and click 'Authorize'")
    print(f"4. Complete the OAuth consent in the browser")
    print(f"5. Wait for the target status to change to READY")
    print(f"")
    print(f"⚠️  DO NOT proceed to the next cell until the target is READY.")
    print("=" * 60)


## Step 4: Authenticate with PKCE

Open a browser for Cognito login using PKCE (no client secret needed). The script starts a local server on port 3001 to capture the auth code, then exchanges it for an access token.

In [ ]:
import http.server
import threading
import webbrowser

# Generate PKCE challenge
code_verifier = secrets.token_urlsafe(64)[:128]
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).rstrip(b'=').decode()

# One-shot callback server
auth_code_result = {}

class PKCEHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        qs = urllib.parse.parse_qs(urllib.parse.urlparse(self.path).query)
        auth_code_result['code'] = qs.get('code', [None])[0]
        self.send_response(200)
        self.send_header('Content-Type', 'text/html')
        self.end_headers()
        self.wfile.write(b"""<html><body style="font-family:system-ui;text-align:center;padding:60px;background:#f0fdf4">
<div style="display:inline-block;padding:40px 60px;border-radius:12px;background:white;box-shadow:0 2px 12px rgba(0,0,0,0.1);max-width:600px;text-align:left">
<h1 style="color:#16a34a;margin-bottom:8px;text-align:center">&#x2714; PKCE Login Successful</h1>
<p style="color:#444;font-size:14px;line-height:1.6"><strong>Flow:</strong> PKCE Authentication &mdash; Browser Login</p>
<p style="color:#444;font-size:14px;line-height:1.6"><strong>What happened:</strong> You logged into Cognito via the browser. The authorization code will be exchanged for an access token using PKCE (no client secret needed). The access token contains your user identity (sub claim) and the app client ID (client_id claim).</p>
<p style="color:#444;font-size:14px;line-height:1.6"><strong>What's next:</strong> The notebook will use this token to call the AgentCore Gateway. If this is your first time accessing GitHub, you will be prompted to authorize in a separate browser window.</p>
<p style="color:#999;font-size:13px;text-align:center;margin-top:20px">You can close this tab.</p>
</div></body></html>""")
    def log_message(self, *a): pass

server = http.server.HTTPServer(('127.0.0.1', 3001), PKCEHandler)
thread = threading.Thread(target=server.handle_request)
thread.start()

# Open browser
authorize_url = (
    f"https://{state['cognito_domain']}.auth.{REGION}.amazoncognito.com/oauth2/authorize"
    f"?response_type=code&client_id={state['public_client_id']}"
    f"&redirect_uri={urllib.parse.quote(PKCE_CALLBACK_URL)}"
    f"&scope=openid+profile+email"
    f"&code_challenge={code_challenge}&code_challenge_method=S256"
)
print(f"Opening browser for login...")
print(f"Login as: {TEST_USER_EMAIL} / {TEST_USER_PASSWORD}")
webbrowser.open(authorize_url)

# Wait for callback
thread.join(timeout=120)
server.server_close()

assert auth_code_result.get('code'), "No auth code received"

# Exchange code for token (PKCE — no secret)
token_url = f"https://{state['cognito_domain']}.auth.{REGION}.amazoncognito.com/oauth2/token"
token_resp = requests.post(token_url, data={
    'grant_type': 'authorization_code',
    'client_id': state['public_client_id'],
    'code': auth_code_result['code'],
    'redirect_uri': PKCE_CALLBACK_URL,
    'code_verifier': code_verifier
}, headers={'Content-Type': 'application/x-www-form-urlencoded'})

tokens = token_resp.json()
ACCESS_TOKEN = tokens['access_token']

# Decode and display
import base64 as b64
payload = ACCESS_TOKEN.split('.')[1]
payload += '=' * (4 - len(payload) % 4)
claims = json.loads(b64.urlsafe_b64decode(payload))
print(f"\n✓ Got access token (PKCE, no secret)")
print(f"  sub:       {claims['sub']}")
print(f"  client_id: {claims['client_id']}")
print(f"  token_use: {claims['token_use']}")

## Step 5: Call Gateway — List Tools

Use the PKCE access token to call the gateway. The gateway validates `client_id` via `allowedClients`.

> **Note:** If no tools are returned, the target may still be syncing. Wait a few minutes and retry — it can take some time for the gateway to discover and cache the tools from the MCP server.

In [ ]:
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {ACCESS_TOKEN}'
}

# Initialize MCP session
init_resp = requests.post(state['gateway_url'], headers=headers, json={
    'jsonrpc': '2.0', 'id': 1, 'method': 'initialize',
    'params': {'protocolVersion': MCP_VERSION, 'capabilities': {}, 'clientInfo': {'name': 'notebook', 'version': '1.0'}}
})
print(f"✓ Initialized: {init_resp.json()['result']['serverInfo']['name']}")

# List tools
headers['Mcp-Protocol-Version'] = MCP_VERSION
tools_resp = requests.post(state['gateway_url'], headers=headers, json={
    'jsonrpc': '2.0', 'id': 2, 'method': 'tools/list', 'params': {}
})
tools = tools_resp.json()['result']['tools']
print(f"✓ {len(tools)} tools available")
for t in tools[:5]:
    print(f"  - {t['name']}")

## Step 6: Call Tool — Handle 3LO Consent

The first tool call triggers a 3LO elicitation (error `-32042`). The user must consent on GitHub, then the callback server binds the session. After that, the tool call succeeds.

In [ ]:
# Start callback server for 3LO binding
callback_proc = subprocess.Popen(
    [sys.executable, 'oauth2_callback_server.py', REGION, '3000', '/tmp/gateway-jwt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(1)

# Save token for callback binding
with open('/tmp/gateway-jwt', 'w') as f:
    f.write(ACCESS_TOKEN)

# Call get_me
tool_name = f"{state['github_target_name']}___get_me"
resp = requests.post(state['gateway_url'], headers=headers, json={
    'jsonrpc': '2.0', 'id': 3, 'method': 'tools/call',
    'params': {'name': tool_name, 'arguments': {}}
})
result = resp.json()

if result.get('error', {}).get('code') == -32042:
    elicit_url = result['error']['data']['elicitations'][0]['url']
    print("⚠ GitHub 3LO consent required (first time for this user)")
    print(f"  Opening browser...")
    webbrowser.open(elicit_url)
    input("  Press Enter after authorizing on GitHub...")
    
    # Retry
    time.sleep(2)
    resp = requests.post(state['gateway_url'], headers=headers, json={
        'jsonrpc': '2.0', 'id': 4, 'method': 'tools/call',
        'params': {'name': tool_name, 'arguments': {}}
    })
    result = resp.json()

# Display result
if 'result' in result:
    data = json.loads(result['result']['content'][0]['text'])
    print(f"\n✓ GitHub get_me succeeded!")
    print(json.dumps(data, indent=2))
else:
    print(f"✖ Error: {result}")

# Stop callback server
callback_proc.terminate()
os.remove('/tmp/gateway-jwt')

## Step 7: Cleanup

Delete all resources created by this tutorial.

In [ ]:
# Delete target
try:
    agentcore_cp.delete_gateway_target(
        gatewayIdentifier=state['gateway_id'],
        targetId=state['github_target_id']
    )
    print(f"✓ Deleted target: {state['github_target_id']}")
    time.sleep(5)
except Exception as e:
    print(f"⚠ {e}")

# Delete gateway
try:
    agentcore_cp.delete_gateway(gatewayIdentifier=state['gateway_id'])
    print(f"✓ Deleted gateway: {state['gateway_id']}")
except Exception as e:
    print(f"⚠ {e}")

# Delete credential provider
try:
    agentcore_cp.delete_oauth2_credential_provider(name=state['github_provider_name'])
    print(f"✓ Deleted provider: {state['github_provider_name']}")
except Exception as e:
    print(f"⚠ {e}")

# Delete Cognito
try:
    cognito.delete_user_pool_domain(UserPoolId=state['pool_id'], Domain=state['cognito_domain'])
    cognito.delete_user_pool(UserPoolId=state['pool_id'])
    print(f"✓ Deleted Cognito pool: {state['pool_id']}")
except Exception as e:
    print(f"⚠ {e}")

print("\n✓ Cleanup complete")

# Delete IAM role
try:
    iam.delete_role_policy(RoleName=state['role_name'], PolicyName='AgentCoreGatewayPolicy')
    iam.delete_role(RoleName=state['role_name'])
    print(f"✓ Deleted IAM role: {state['role_name']}")
except Exception as e:
    print(f"⚠ IAM role: {e}")
